## Text cleaning and Normalization

In [1]:
import re
import pandas as pd
import numpy as np

# Load Data
file_path = '/Users/denghaonan/Desktop/capstone/code/NTSB_pipeline/ntsb_with_extracted.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please check the path.")

def clean_text(text):
    if pd.isna(text):
        return ""
    
    # Convert to string and lower case for processing 
    text = str(text).lower()
    
    # Remove special characters / non-ascii but keep punctuation relevant for sentence splitting
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # Remove non-ASCII
    
    return text.strip()

# Apply cleaning
df['clean_text'] = df['fulltext'].apply(clean_text)
print("\nText cleaning complete.")

Dataset loaded successfully.

Text cleaning complete.


## Segmentation + Tokenization

In [5]:
import spacy

# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

def tokenize_text(text):
    if not text:
        return []
    # Process text with spacy (segmentation + tokenization)
    doc = nlp(text)
    return doc

df['spacy_doc'] = list(nlp.pipe(df['clean_text']))

## Supervised Learning Extraction

In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn import metrics

# --- Configuration ---
# List of columns you want to fill using the NLP model.
# These should be categorical or short-text columns.
target_columns_to_fill = [
    'Findings', 
    'ProbableCause',
    'AirportID',
    'PurposeOfFlight',
    'WeatherCondition ',
    'HighestInjuryLevel',
    'AirCraftDamage',
    'NumberOfEngines',
    'Condition_of_Light'
]


# --- The "Universal" Train-Predict Function ---
def fill_missing_with_ml(dataframe, target_col, text_col='clean_text'):
    """
    1. Splits data into TRAIN (rows where target_col exists) and PREDICT (rows where target_col is missing).
    2. Vectorizes the text data.
    3. Trains a classifier on the TRAIN set.
    4. Predicts values for the PREDICT set.
    5. Returns the updated series.
    """
    print(f"\n--- Processing Column: {target_col} ---")
    
    # 1. Separation
    # Training data: Rows where we HAVE the target value AND text
    train_mask = dataframe[target_col].notna() & (dataframe[text_col] != "")
    
    # Prediction data: Rows where the target value is MISSING AND text exists
    predict_mask = dataframe[target_col].isna() & (dataframe[text_col] != "")
    
    if predict_mask.sum() == 0:
        print(f"No missing values to fill for {target_col}. Skipping.")
        return dataframe[target_col]
    
    if train_mask.sum() < 50:
        print(f"Not enough training data ({train_mask.sum()} rows) for {target_col}. Skipping.")
        return dataframe[target_col]

    # 2. Prepare Sets
    X_train = dataframe.loc[train_mask, text_col]
    y_train = dataframe.loc[train_mask, target_col].astype(str) # Ensure target is string for classification
    X_predict = dataframe.loc[predict_mask, text_col]

    print(f"Training on {len(X_train)} samples. Predicting for {len(X_predict)} blanks.")

    # 3. Build Pipeline
    # We use TF-IDF for features and SGDClassifier (SVM) which is fast and robust for text
    text_clf = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english', 
            max_features=5000, # Limit vocab to top 5k words to reduce noise
            ngram_range=(1, 2) # Look at unigrams and bigrams 
        )),
        ('clf', SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, random_state=42, max_iter=5, tol=None)),
    ])

    # 4. Train Model
    text_clf.fit(X_train, y_train)
    
    # Evaluate on a small subset of training data 
    score = text_clf.score(X_train, y_train)
    print(f"Model Training Accuracy: {score:.2f}")

    # 5. Predict
    predicted_values = text_clf.predict(X_predict)
    
    # 6. Fill Values
    # We create a copy to avoid SettingWithCopy warnings
    original_series = dataframe[target_col].copy()
    original_series.loc[predict_mask] = predicted_values
    
    print(f"Filled {len(predicted_values)} missing values for {target_col}.")
    
    return original_series

# --- Execution Loop ---

for col in target_columns_to_fill:
    # Check if column exists in your CSV
    if col in df.columns:
        df[col] = fill_missing_with_ml(df, col)
    else:
        print(f"Column {col} not found in dataset. Skipping.")

print("\nAll columns processed.")


--- Processing Column: Findings ---
Training on 1662 samples. Predicting for 224 blanks.
Model Training Accuracy: 0.96
Filled 224 missing values for Findings.

--- Processing Column: ProbableCause ---
Training on 1665 samples. Predicting for 221 blanks.
Model Training Accuracy: 0.99
Filled 221 missing values for ProbableCause.

--- Processing Column: AirportID ---
Training on 1286 samples. Predicting for 600 blanks.
Model Training Accuracy: 0.98
Filled 600 missing values for AirportID.

--- Processing Column: PurposeOfFlight ---
Training on 1797 samples. Predicting for 89 blanks.
Model Training Accuracy: 0.88
Filled 89 missing values for PurposeOfFlight.
Column WeatherCondition  not found in dataset. Skipping.

--- Processing Column: HighestInjuryLevel ---
Training on 766 samples. Predicting for 1120 blanks.
Model Training Accuracy: 0.97
Filled 1120 missing values for HighestInjuryLevel.

--- Processing Column: AirCraftDamage ---
Training on 1844 samples. Predicting for 42 blanks.
Mod

## Output

In [ ]:
# df.to_csv('ntsb_ml_filled.csv', index=False)

## Handle the unknown data

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

# 1. Load the previously filled file
# input_path = 'ntsb_ml_filled.csv'
output_path = 'ntsb_ml_filled.csv'

# try:
#     df = pd.read_csv(input_path)
#     print("Data loaded successfully.")
# except FileNotFoundError:
#     print(f"Error: {input_path} not found.")
#     exit()

# --- Configuration ---
target_columns_to_refine = [
    'Condition_of_Light',
    'Lowest_Cloud_Condition',
    'Wind_Info',
    'Lowest_Ceiling',
    'Visibility'
]

# --- Helper Functions ---

def clean_text_simple(text):
    return str(text).lower().strip() if pd.notnull(text) else ""

def simplify_target_label(val, col_name):
    """
    Simplifies complex strings to make them learnable classes for the ML model.
    """
    val = str(val).strip()
    
    # Logic 1: Handle "Unknown" variations -> convert to NaN so we know to predict it
    if val.lower() in ['unknown', 'unkown', 'nan', '', 'none', '.', '?']:
        return np.nan
    
    # Logic 2: Specific column simplifications
    if col_name in ['Lowest_Cloud_Condition', 'Lowest_Ceiling']:
        # "Scattered / 11000 ft AGL" -> "Scattered"
        if '/' in val:
            return val.split('/')[0].strip()
            
    elif col_name == 'Wind_Info':
        # "5 knots / None" -> "5 knots" (Simplifying to speed/presence)
        # Note: Predicting exact wind speed from text is hard for classifiers, 
        # but this groups them better than full strings.
        if '/' in val:
            return val.split('/')[0].strip()

    elif col_name == 'Visibility':
        # Keep standard "10 miles", "3 miles"
        pass
        
    return val

# --- The ML Fill Function ---

def fill_complex_cols_with_ml(dataframe, target_col, text_col='clean_text'):
    print(f"\n--- Processing Column: {target_col} ---")
    
    # 1. Pre-process the target column to handle "Unknown" and simplify structure
    # We create a temporary series for training to avoid overwriting raw data permanently yet
    temp_target = dataframe[target_col].apply(lambda x: simplify_target_label(x, target_col))
    
    # 2. Separation
    train_mask = temp_target.notna() & (dataframe[text_col] != "")
    predict_mask = temp_target.isna() & (dataframe[text_col] != "")
    
    if predict_mask.sum() == 0:
        print(f"No 'Unknown' values found to fill for {target_col}. Skipping.")
        return dataframe[target_col]
    
    # Lower threshold slightly as some simplified classes might still be rare
    if train_mask.sum() < 10: 
        print(f"Not enough training data ({train_mask.sum()} rows) for {target_col}. Skipping.")
        return dataframe[target_col]

    # 3. Prepare Sets
    X_train = dataframe.loc[train_mask, text_col]
    y_train = temp_target[train_mask].astype(str)
    X_predict = dataframe.loc[predict_mask, text_col]

    print(f"Training on {len(X_train)} samples. Predicting for {len(X_predict)} blanks.")
    print(f"Unique classes found: {y_train.nunique()} (Examples: {y_train.unique()[:3]})")

    # 4. Build Pipeline
    # Using SGDClassifier (SVM) optimized for multi-class text classification
    text_clf = Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english', 
            max_features=5000, 
            ngram_range=(1, 2)
        )),
        ('clf', SGDClassifier(
            loss='hinge', 
            penalty='l2', 
            alpha=1e-3, 
            random_state=42, 
            max_iter=10, 
            tol=1e-3
        )),
    ])

    # 5. Train
    try:
        text_clf.fit(X_train, y_train)
    except ValueError as e:
        print(f"Skipping {target_col} due to training error (likely only 1 class): {e}")
        return dataframe[target_col]

    # 6. Predict
    predicted_values = text_clf.predict(X_predict)
    
    # 7. Update DataFrame
    # We overwrite the "Unknown" values with the predicted simplified values
    original_series = dataframe[target_col].copy()
    
    # We need to map the predicted values back to the original rows
    # Note: The predictions will be the SIMPLIFIED version (e.g., "Scattered")
    # This is preferred because we cannot accurately hallucinate the altitude numbers.
    original_series.loc[predict_mask] = predicted_values
    
    print(f"Filled {len(predicted_values)} values.")
    return original_series

# --- Execution ---

# Ensure clean text exists
df['clean_text'] = df['fulltext'].apply(clean_text_simple)

for col in target_columns_to_refine:
    if col in df.columns:
        df[col] = fill_complex_cols_with_ml(df, col)
    else:
        print(f"Column {col} missing.")

# --- Save ---
df.to_csv(output_path, index=False)
print(f"\nProcessing complete. Saved to {output_path}")


--- Processing Column: Condition_of_Light ---
No 'Unknown' values found to fill for Condition_of_Light. Skipping.

--- Processing Column: Lowest_Cloud_Condition ---
Training on 1524 samples. Predicting for 362 blanks.
Unique classes found: 11 (Examples: <StringArray>
['Clear', 'Few', 'Scattered']
Length: 3, dtype: str)


/Users/denghaonan/Desktop/capstone/code/.venv/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


Filled 362 values.

--- Processing Column: Wind_Info ---
Training on 169 samples. Predicting for 1717 blanks.
Unique classes found: 26 (Examples: <StringArray>
['13 knots', '3 knots', '10 knots']
Length: 3, dtype: str)


/Users/denghaonan/Desktop/capstone/code/.venv/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


Filled 1717 values.

--- Processing Column: Lowest_Ceiling ---
Training on 554 samples. Predicting for 1332 blanks.
Unique classes found: 8 (Examples: <StringArray>
['Overcast', 'IndefiniteVV', 'Broken']
Length: 3, dtype: str)


/Users/denghaonan/Desktop/capstone/code/.venv/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


Filled 1332 values.

--- Processing Column: Visibility ---
Training on 197 samples. Predicting for 1689 blanks.
Unique classes found: 13 (Examples: <StringArray>
['10 miles', '1 miles', '30 miles']
Length: 3, dtype: str)


/Users/denghaonan/Desktop/capstone/code/.venv/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


Filled 1689 values.

Processing complete. Saved to ntsb_ml_filled.csv
